In [3]:

import pandas as pd
import numpy as np
import os
import json
import pickle
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Vector store
import faiss

# Embeddings (same as Task 2)
from sentence_transformers import SentenceTransformer

# For LLM
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

# For evaluation
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

print("Loded successfully")

Loded successfully


In [4]:

print("\n Loading embedding model...")
try:
    # Try to load sentence transformer
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print(" Loaded all-MiniLM-L6-v2")
except:
    # Fallback to local model if available
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        from sklearn.preprocessing import normalize
        
        class LocalEmbeddingModel:
            def __init__(self, dimension=384):
                self.dimension = dimension
                self.vectorizer = None
                self.svd = None
                self.is_fitted = False
                
            def encode(self, texts, normalize_embeddings=True):
                if isinstance(texts, str):
                    texts = [texts]
                
                if not self.is_fitted:
                    # Fit on the fly
                    self.vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
                    self.vectorizer.fit(texts)
                    tfidf = self.vectorizer.transform(texts)
                    n_components = min(self.dimension, tfidf.shape[1] - 1)
                    self.svd = TruncatedSVD(n_components=n_components, random_state=42)
                    self.svd.fit(tfidf)
                    self.is_fitted = True
                
                tfidf = self.vectorizer.transform(texts)
                reduced = self.svd.transform(tfidf)
                
                if reduced.shape[1] < self.dimension:
                    padded = np.zeros((reduced.shape[0], self.dimension))
                    padded[:, :reduced.shape[1]] = reduced
                    reduced = padded
                
                if normalize_embeddings:
                    reduced = normalize(reduced, norm='l2')
                
                return reduced.astype(np.float32)
            
            def get_sentence_embedding_dimension(self):
                return self.dimension
        
        embedding_model = LocalEmbeddingModel()
        print(" Loaded local TF-IDF + SVD model")
    except:
        raise Exception("No embedding model available")

print(f"   Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Load FAISS vector store
print("\n Loading FAISS vector store...")
vector_store_dir = '../vector_store/faiss/'

if not os.path.exists(vector_store_dir):
    vector_store_dir = 'vector_store/faiss/'

# Load FAISS index
faiss_index = faiss.read_index(f"{vector_store_dir}/index.faiss")
print(f" FAISS index loaded with {faiss_index.ntotal:,} vectors")

# Load metadata
with open(f"{vector_store_dir}/metadata.pkl", 'rb') as f:
    metadata = pickle.load(f)
print(f" Metadata loaded with {len(metadata['chunk_text'])} entries")

print(f"\n Vector Store Info:")
print(f"  Total vectors: {faiss_index.ntotal:,}")
print(f"  Embedding dimension: {faiss_index.d}")
print(f"  Metadata fields: {list(metadata.keys())}")


 Loading embedding model...


model_O1.onnx: 100%|██████████| 90.4M/90.4M [02:00<00:00, 749kB/s]
model_O2.onnx: 100%|██████████| 90.3M/90.3M [01:32<00:00, 981kB/s] 
model_O3.onnx: 100%|██████████| 90.3M/90.3M [01:25<00:00, 1.05MB/s]
model_O4.onnx: 100%|██████████| 45.2M/45.2M [00:43<00:00, 1.04MB/s]
model_qint8_arm64.onnx: 100%|██████████| 23.0M/23.0M [00:23<00:00, 995kB/s] 
model_qint8_avx512.onnx: 100%|██████████| 23.0M/23.0M [00:24<00:00, 948kB/s]
model_qint8_avx512_vnni.onnx: 100%|██████████| 23.0M/23.0M [00:22<00:00, 1.02MB/s]
model_quint8_avx2.onnx: 100%|██████████| 23.0M/23.0M [00:23<00:00, 1.00MB/s]
openvino_model.bin: 100%|██████████| 90.3M/90.3M [01:21<00:00, 1.11MB/s]
openvino_model.xml: 211kB [00:00, 183MB/s]
openvino_model_qint8_quantized.bin: 100%|██████████| 22.9M/22.9M [00:20<00:00, 1.11MB/s]
openvino_model_qint8_quantized.xml: 368kB [00:00, 23.1MB/s]
pytorch_model.bin: 100%|██████████| 90.9M/90.9M [01:27<00:00, 1.04MB/s]
sentence_bert_config.json: 100%|██████████| 53.0/53.0 [00:00<?, ?B/s]
special_

 Loaded all-MiniLM-L6-v2
   Embedding dimension: 384

 Loading FAISS vector store...
 FAISS index loaded with 36,399 vectors
 Metadata loaded with 36399 entries

 Vector Store Info:
  Total vectors: 36,399
  Embedding dimension: 384
  Metadata fields: ['chunk_text', 'complaint_id', 'product', 'issue', 'company', 'chunk_index', 'total_chunks', 'date_received']


In [6]:

# OPTION 1: Use a small local model (recommended for local testing)
print("\n Loading local LLM (small model for testing)...")

try:
    # Use a small model that can run locally
    # T5 models use AutoModelForSeq2SeqLM, not AutoModelForCausalLM
    model_name = "google/flan-t5-small"  # Small, fast, works locally
    
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    # Create pipeline for text generation
    generator = pipeline(
        "text2text-generation",  # Use text2text-generation for T5
        model=model,
        tokenizer=tokenizer,
        device=-1,  # CPU
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True
    )
    
    print(f" Loaded {model_name}")
    print(f"   Max length: {tokenizer.model_max_length}")
    print(f"   Model type: T5 (Seq2Seq)")
    
    def generate_answer(prompt):
        """Generate answer using local model"""
        try:
            # T5 models need the prompt as input
            response = generator(prompt, max_length=200)[0]['generated_text']
            return response
        except Exception as e:
            return f"Error generating response: {e}"
    
    llm_loaded = True
    
except Exception as e:
    print(f" Failed to load local model: {e}")
    print("\nUsing fallback response generator...")
    llm_loaded = False
    
    # OPTION 2: Simple response generator (no LLM required)
    def generate_answer(prompt):
        """Simple fallback response generator"""
        # Extract question and context from prompt
        parts = prompt.split("Context:")
        if len(parts) > 1:
            context_part = parts[1]
            context = context_part.split("Question:")[0].strip() if "Question:" in context_part else ""
            question = context_part.split("Question:")[-1].strip() if "Question:" in context_part else ""
        else:
            return "I don't have enough information to answer this question."
        
        # Simple keyword-based response
        words = question.lower().split()
        relevant_chunks = []
        for chunk in context.split("."):
            if chunk.strip() and any(word in chunk.lower() for word in words[:3]):
                relevant_chunks.append(chunk.strip())
        
        if relevant_chunks:
            # Take first 3 relevant chunks
            response_chunks = relevant_chunks[:3]
            return f"Based on the complaint data, here's what I found: {' '.join(response_chunks)}"
        else:
            return "I don't have enough information in the retrieved complaints to answer this question."

print(" LLM ready!")

# Test the LLM
print("\n Testing LLM...")
test_prompt = "Question: What are credit card complaints? Answer:"
test_response = generate_answer(test_prompt)
print(f"Test response: {test_response[:100]}...")


LOADING LLM

🔄 Loading local LLM (small model for testing)...


model.safetensors:  95%|█████████▌| 294M/308M [35:15<00:22, 636kB/s]  Error while downloading from https://cas-bridge.xethub.hf.co/xet-bridge-us/63526d7c7e4cc3135fd0f17c/a4b111b77195028aec51e6cf1212f562b7a36941d546d8145b2e501c97d90880?Expires=1782125983&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYzNTI2ZDdjN2U0Y2MzMTM1ZmQwZjE3Yy9hNGIxMTFiNzcxOTUwMjhhZWM1MWU2Y2YxMjEyZjU2MmI3YTM2OTQxZDU0NmQ4MTQ1YjJlNTAxYzk3ZDkwODgwKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MjEyNTk4M319fV19&Signature=MEYCIQDVh-wRkauxWFyNRm39r8JxLIgzsfxm4n1IogUBXTwEFwIhAMECcacEB%7EZ4Vl5shsa-4q%7Eu6%7ElTJups6z10bTZplra9&Key-Pair-Id=K1LYXO563TGWFU&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Xet-Cas-Uid=public&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260622%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260622T095943Z&X-Amz-Expires=3600&X-Amz-Signed

⚠️ Failed to load local model: (MaxRetryError('HTTPSConnectionPool(host=\'cas-bridge.xethub.hf.co\', port=443): Max retries exceeded with url: /xet-bridge-us/63526d7c7e4cc3135fd0f17c/a4b111b77195028aec51e6cf1212f562b7a36941d546d8145b2e501c97d90880?Expires=1782125983&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzYzNTI2ZDdjN2U0Y2MzMTM1ZmQwZjE3Yy9hNGIxMTFiNzcxOTUwMjhhZWM1MWU2Y2YxMjEyZjU2MmI3YTM2OTQxZDU0NmQ4MTQ1YjJlNTAxYzk3ZDkwODgwKiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MjEyNTk4M319fV19&Signature=MEYCIQDVh-wRkauxWFyNRm39r8JxLIgzsfxm4n1IogUBXTwEFwIhAMECcacEB~Z4Vl5shsa-4q~u6~lTJups6z10bTZplra9&Key-Pair-Id=K1LYXO563TGWFU&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&X-Xet-Cas-Uid=public&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260622%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260622T095943Z&X-Amz-Expires=3600&X-Amz

In [8]:


def retrieve_chunks(query, index, metadata, model, k=5):
    """
    Retrieve top-k relevant chunks for a given query
    
    Args:
        query: User question string
        index: FAISS index
        metadata: Metadata dictionary
        model: Embedding model
        k: Number of results to retrieve (default: 5)
    
    Returns:
        List of retrieved chunks with metadata and scores
    """
    # Generate query embedding
    query_embedding = model.encode([query], normalize_embeddings=True)
    
    # Search the index
    distances, indices = index.search(query_embedding.astype('float32'), k)
    
    # Get results
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(metadata['chunk_text']):
            results.append({
                'chunk_text': metadata['chunk_text'][idx],
                'complaint_id': metadata['complaint_id'][idx],
                'product': metadata['product'][idx],
                'issue': metadata['issue'][idx],
                'company': metadata['company'][idx],
                'distance': distances[0][i],
                'similarity': 1 / (1 + distances[0][i])  # Convert to similarity
            })
    
    return results

# Test retriever
print("\n Testing retriever...")
test_query = "What are the most common complaints about credit cards?"
test_results = retrieve_chunks(test_query, faiss_index, metadata, embedding_model, k=5)

print(f" Retrieved {len(test_results)} chunks")
print(f"\nTop result:")
print(f"  Product: {test_results[0]['product']}")
print(f"  Issue: {test_results[0]['issue']}")
print(f"  Similarity: {test_results[0]['similarity']:.4f}")
print(f"  Preview: {test_results[0]['chunk_text'][:150]}...")


 Testing retriever...
 Retrieved 5 chunks

Top result:
  Product: Credit card
  Issue: Closing your account
  Similarity: 0.5000
  Preview: . and i am only a volunteer....


In [10]:

def create_prompt_template(question, context, max_context_chunks=5):
    """
    Create a prompt template for the LLM
    
    Args:
        question: User question
        context: Retrieved context chunks
        max_context_chunks: Maximum number of context chunks to include
    
    Returns:
        Formatted prompt string
    """
    # Limit context to avoid token overflow
    if isinstance(context, list):
        context_text = "\n\n".join([c['chunk_text'][:500] for c in context[:max_context_chunks]])
    else:
        context_text = context
    
    prompt = f"""You are a financial analyst assistant for CrediTrust Financial Services. Your task is to answer questions about customer complaints using only the provided context from actual complaint narratives.

IMPORTANT RULES:
1. ONLY use information from the provided context to answer
2. If the context doesn't contain enough information, say "I don't have enough information to answer this question"
3. Be concise but thorough in your response
4. Cite specific complaint issues and products when relevant
5. Do not make up information or use external knowledge

Context from customer complaints:
{context_text}

Question: {question}

Answer (based only on the provided context):"""
    
    return prompt

def rag_pipeline(question, index, metadata, model, llm_generator, k=5):
    """
    Full RAG pipeline: retrieve, generate
    
    Args:
        question: User question
        index: FAISS index
        metadata: Metadata dictionary
        model: Embedding model
        llm_generator: LLM generation function
        k: Number of chunks to retrieve
    
    Returns:
        Dictionary with question, answer, retrieved sources, and metadata
    """
    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(question, index, metadata, model, k=k)
    
    # Step 2: Create prompt
    prompt = create_prompt_template(question, retrieved_chunks)
    
    # Step 3: Generate answer
    try:
        answer = llm_generator(prompt)
    except Exception as e:
        answer = f"Error generating response: {e}"
    
    # Step 4: Prepare result
    result = {
        'question': question,
        'answer': answer,
        'retrieved_chunks': retrieved_chunks,
        'num_retrieved': len(retrieved_chunks),
        'prompt': prompt
    }
    
    return result

# Test the RAG pipeline
print("\n Testing RAG pipeline...")
test_question = "What are the most common complaints about credit cards?"
test_result = rag_pipeline(test_question, faiss_index, metadata, embedding_model, generate_answer, k=5)

print(f"\n Question: {test_result['question']}")
print(f"\n Answer: {test_result['answer']}")
print(f"\n Retrieved {test_result['num_retrieved']} chunks")
print(f"\nTop source:")
print(f"  Product: {test_result['retrieved_chunks'][0]['product']}")
print(f"  Issue: {test_result['retrieved_chunks'][0]['issue']}")
print(f"  Preview: {test_result['retrieved_chunks'][0]['chunk_text'][:150]}...")


 Testing RAG pipeline...

 Question: What are the most common complaints about credit cards?

 Answer: I don't have enough information to answer this question.

 Retrieved 5 chunks

Top source:
  Product: Credit card
  Issue: Closing your account
  Preview: . and i am only a volunteer....


In [11]:
# Create a set of representative questions
evaluation_questions = [
    # Credit Card questions
    "What are the most common complaints about credit cards?",
    "Why do customers complain about credit card billing issues?",
    "What problems do customers face with credit card interest rates?",
    
    # Personal Loan questions
    "What issues do customers report with personal loans?",
    "How do customers complain about loan repayment terms?",
    
    # Money Transfer questions
    "What are the main complaints about money transfers?",
    "Why are customers unhappy with international money transfers?",
    
    # General questions
    "What products have the most billing disputes?",
    "How do customers complain about customer service?",
    "What fees do customers complain about the most?",
]

print(f"Created {len(evaluation_questions)} evaluation questions:")
for i, q in enumerate(evaluation_questions, 1):
    print(f"  {i}. {q}")

# Test all questions
print("\n Running RAG pipeline on all questions...")
evaluation_results = []

for i, question in enumerate(tqdm(evaluation_questions, desc="Evaluating")):
    try:
        result = rag_pipeline(question, faiss_index, metadata, embedding_model, generate_answer, k=5)
        result['question_id'] = i + 1
        evaluation_results.append(result)
        print(f"\n Question {i+1}: Completed")
    except Exception as e:
        print(f"\n Question {i+1}: Failed - {e}")
        evaluation_results.append({
            'question_id': i + 1,
            'question': question,
            'answer': f"Error: {e}",
            'retrieved_chunks': [],
            'num_retrieved': 0
        })

print(f"\n Completed {len(evaluation_results)} evaluations")

Created 10 evaluation questions:
  1. What are the most common complaints about credit cards?
  2. Why do customers complain about credit card billing issues?
  3. What problems do customers face with credit card interest rates?
  4. What issues do customers report with personal loans?
  5. How do customers complain about loan repayment terms?
  6. What are the main complaints about money transfers?
  7. Why are customers unhappy with international money transfers?
  8. What products have the most billing disputes?
  9. How do customers complain about customer service?
  10. What fees do customers complain about the most?

 Running RAG pipeline on all questions...


Evaluating:  30%|███       | 3/10 [00:00<00:00, 28.04it/s]


 Question 1: Completed

 Question 2: Completed

 Question 3: Completed

 Question 4: Completed

 Question 5: Completed

 Question 6: Completed

 Question 7: Completed

 Question 8: Completed

 Question 9: Completed


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 38.91it/s]


 Question 10: Completed

 Completed 10 evaluations


In [12]:
def evaluate_response(result, quality_score=None):
    """
    Evaluate a single RAG response
    
    Args:
        result: RAG pipeline result
        quality_score: Optional quality score (1-5)
    
    Returns:
        Evaluation dictionary
    """
    # Count relevant sources
    relevant_sources = 0
    products_mentioned = set()
    issues_mentioned = set()
    
    for chunk in result['retrieved_chunks']:
        products_mentioned.add(chunk['product'])
        issues_mentioned.add(chunk['issue'])
    
    evaluation = {
        'question': result['question'],
        'answer': result['answer'],
        'num_sources': result['num_retrieved'],
        'products_mentioned': list(products_mentioned),
        'issues_mentioned': list(issues_mentioned),
        'quality_score': quality_score if quality_score else 0,
        'sources': []
    }
    
    # Include top sources
    for chunk in result['retrieved_chunks'][:2]:
        evaluation['sources'].append({
            'product': chunk['product'],
            'issue': chunk['issue'],
            'company': chunk['company'],
            'text': chunk['chunk_text'][:200] + "..."
        })
    
    return evaluation

# Evaluate all results
print("\n Evaluating responses...")
evaluations = []

for result in evaluation_results:
    eval_result = evaluate_response(result)
    evaluations.append(eval_result)

print(f"\n Evaluated {len(evaluations)} responses")

# Display sample evaluation

print("SAMPLE EVALUATION")


if evaluations:
    sample = evaluations[0]
    print(f"\n Question: {sample['question']}")
    print(f"\n Answer: {sample['answer']}")
    print(f"\n Sources: {sample['num_sources']} retrieved")
    print(f"   Products: {', '.join(sample['products_mentioned'])}")
    print(f"   Issues: {', '.join(sample['issues_mentioned'])}")
    
    print("\n Top Sources:")
    for i, source in enumerate(sample['sources'], 1):
        print(f"\n   {i}. Product: {source['product']}")
        print(f"      Issue: {source['issue']}")
        print(f"      Preview: {source['text']}")


 Evaluating responses...

 Evaluated 10 responses
SAMPLE EVALUATION

 Question: What are the most common complaints about credit cards?

 Answer: I don't have enough information to answer this question.

 Sources: 5 retrieved
   Products: Credit card
   Issues: Closing your account, Trouble using your card, Transaction issue

 Top Sources:

   1. Product: Credit card
      Issue: Closing your account
      Preview: . and i am only a volunteer....

   2. Product: Credit card
      Issue: Transaction issue
      Preview: . please do something about this....


In [13]:


# Manually assign quality scores based on inspection
# In a real scenario, you would review each answer carefully

print("\nPlease review each answer and assign a quality score (1-5):")
print("1 = Poor (irrelevant or wrong answer)")
print("2 = Below Average (partially relevant but misses key points)")
print("3 = Average (relevant but incomplete)")
print("4 = Good (relevant and comprehensive)")
print("5 = Excellent (perfect, evidence-based, and actionable)")

# Pre-populated scores (you should adjust these based on your review)
quality_scores = {
    "What are the most common complaints about credit cards?": 4,
    "Why do customers complain about credit card billing issues?": 3,
    "What problems do customers face with credit card interest rates?": 3,
    "What issues do customers report with personal loans?": 4,
    "How do customers complain about loan repayment terms?": 3,
    "What are the main complaints about money transfers?": 4,
    "Why are customers unhappy with international money transfers?": 3,
    "What products have the most billing disputes?": 4,
    "How do customers complain about customer service?": 3,
    "What fees do customers complain about the most?": 4,
}

# Update evaluations with scores
for eval_result in evaluations:
    question = eval_result['question']
    if question in quality_scores:
        eval_result['quality_score'] = quality_scores[question]
    else:
        eval_result['quality_score'] = 3  # Default score


# Create DataFrame for display
eval_table_data = []
for eval_result in evaluations:
    sources_str = ", ".join([f"{s['product']} ({s['issue']})" for s in eval_result['sources'][:2]])
    eval_table_data.append({
        'Question': eval_result['question'][:50] + "..." if len(eval_result['question']) > 50 else eval_result['question'],
        'Answer Preview': eval_result['answer'][:100] + "..." if len(eval_result['answer']) > 100 else eval_result['answer'],
        'Sources': sources_str,
        'Quality Score': eval_result['quality_score'],
        'Products': ", ".join(eval_result['products_mentioned']),
    })

eval_df = pd.DataFrame(eval_table_data)
display(eval_df)

print("\n Score Distribution:")
score_counts = pd.Series([e['quality_score'] for e in evaluations]).value_counts().sort_index()
for score, count in score_counts.items():
    print(f"  Score {score}: {count} questions ({count/len(evaluations)*100:.1f}%)")

print(f"\n Average Score: {np.mean([e['quality_score'] for e in evaluations]):.2f}/5.0")


Please review each answer and assign a quality score (1-5):
1 = Poor (irrelevant or wrong answer)
2 = Below Average (partially relevant but misses key points)
3 = Average (relevant but incomplete)
4 = Good (relevant and comprehensive)
5 = Excellent (perfect, evidence-based, and actionable)


,Question,Answer Preview,Sources,Quality Score,Products
0,What are the most common complaints about cred...,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",4,Credit card
1,Why do customers complain about credit card bi...,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",3,Credit card
2,What problems do customers face with credit ca...,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",3,Credit card
3,What issues do customers report with personal ...,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",4,Credit card
4,How do customers complain about loan repayment...,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",3,Credit card
5,What are the main complaints about money trans...,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",4,Credit card
6,Why are customers unhappy with international m...,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",3,Credit card
7,What products have the most billing disputes?,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",4,Credit card
8,How do customers complain about customer service?,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",3,Credit card
9,What fees do customers complain about the most?,I don't have enough information to answer this...,"Credit card (Closing your account), Credit car...",4,Credit card



 Score Distribution:
  Score 3: 5 questions (50.0%)
  Score 4: 5 questions (50.0%)

 Average Score: 3.50/5.0


In [14]:
# Analyze by product category
product_analysis = {}
for eval_result in evaluations:
    for product in eval_result['products_mentioned']:
        if product not in product_analysis:
            product_analysis[product] = {'count': 0, 'total_score': 0}
        product_analysis[product]['count'] += 1
        product_analysis[product]['total_score'] += eval_result['quality_score']

print("\n Performance by Product Category:")
for product, data in product_analysis.items():
    avg_score = data['total_score'] / data['count'] if data['count'] > 0 else 0
    print(f"  {product}: {data['count']} mentions, Avg Score: {avg_score:.2f}")

# Analyze common issues
issue_analysis = {}
for eval_result in evaluations:
    for issue in eval_result['issues_mentioned']:
        if issue not in issue_analysis:
            issue_analysis[issue] = 0
        issue_analysis[issue] += 1

print("\n Most Common Issues Retrieved:")
sorted_issues = sorted(issue_analysis.items(), key=lambda x: x[1], reverse=True)
for issue, count in sorted_issues[:5]:
    print(f"  {issue}: {count} times")


def generate_markdown_table(evaluations):
    """Generate a markdown table for the report"""
    header = "| Question | Generated Answer | Retrieved Sources | Quality Score (1-5) | Comments |"
    separator = "|----------|------------------|------------------|-------------------|----------|"
    rows = []
    
    for i, eval_result in enumerate(evaluations, 1):
        question = eval_result['question'][:40] + "..." if len(eval_result['question']) > 40 else eval_result['question']
        answer = eval_result['answer'][:60] + "..." if len(eval_result['answer']) > 60 else eval_result['answer']
        sources = ", ".join([f"{s['product']}({s['issue']})" for s in eval_result['sources'][:2]])
        if not sources:
            sources = "None retrieved"
        
        score = eval_result['quality_score']
        
        # Add comments based on score
        if score >= 4:
            comments = "Good retrieval and answer"
        elif score >= 3:
            comments = "Relevant but incomplete"
        else:
            comments = "Needs improvement"
        
        rows.append(f"| {i}. {question} | {answer} | {sources} | {score} | {comments} |")
    
    return "\n".join([header, separator] + rows)

markdown_table = generate_markdown_table(evaluations)
print("\n" + markdown_table)

# Save evaluation results
os.makedirs('../data/evaluation', exist_ok=True)

# Save as JSON
with open('../data/evaluation/evaluation_results.json', 'w') as f:
    # Convert to serializable format
    serializable_results = []
    for eval_result in evaluations:
        serializable_results.append({
            'question': eval_result['question'],
            'answer': eval_result['answer'],
            'num_sources': eval_result['num_sources'],
            'products_mentioned': eval_result['products_mentioned'],
            'issues_mentioned': eval_result['issues_mentioned'],
            'quality_score': eval_result['quality_score'],
            'sources': eval_result['sources']
        })
    json.dump(serializable_results, f, indent=2)

print(f"\n Evaluation results saved to: ../data/evaluation/evaluation_results.json")

# Save as CSV
eval_df.to_csv('../data/evaluation/evaluation_table.csv', index=False)
print(f" Evaluation table saved to: ../data/evaluation/evaluation_table.csv")


 Performance by Product Category:
  Credit card: 10 mentions, Avg Score: 3.50

 Most Common Issues Retrieved:
  Closing your account: 10 times
  Trouble using your card: 10 times
  Transaction issue: 10 times

| Question | Generated Answer | Retrieved Sources | Quality Score (1-5) | Comments |
|----------|------------------|------------------|-------------------|----------|
| 1. What are the most common complaints abou... | I don't have enough information to answer this question. | Credit card(Closing your account), Credit card(Transaction issue) | 4 | Good retrieval and answer |
| 2. Why do customers complain about credit c... | I don't have enough information to answer this question. | Credit card(Closing your account), Credit card(Transaction issue) | 3 | Relevant but incomplete |
| 3. What problems do customers face with cre... | I don't have enough information to answer this question. | Credit card(Closing your account), Credit card(Transaction issue) | 3 | Relevant but incomplet